# VidyaAI: Fine-tuning Gemma 4 E4B with QLoRA

Multilingual education assistant for Indian students.

**Runtime:** Colab Pro — **A100 GPU + High-RAM**

**Pipeline:**
1. Install pinned versions
2. HuggingFace login (paste token)
3. Load Gemma 4 E4B in 4-bit
4. Apply LoRA adapter
5. Build dataset (Hindi + English educational QA)
6. Train (~45-60 min on A100)
7. Test
8. Push LoRA to HF Hub

In [ ]:
# Pin verified versions for Gemma 4 support
!pip install --quiet --upgrade "transformers>=5.3.0" "trl>=1.0.0" "peft>=0.18.0" "accelerate>=1.0.0" "bitsandbytes>=0.49.0"
!pip install --quiet --upgrade datasets huggingface_hub sentencepiece protobuf

import transformers, trl, peft
print(f"transformers: {transformers.__version__}")
print(f"trl: {trl.__version__}")
print(f"peft: {peft.__version__}")

In [ ]:
# HuggingFace authentication — REQUIRED to download Gemma 4 (gated model)
# Get token from: https://huggingface.co/settings/tokens
import os
from getpass import getpass

# Try Colab secret first (recommended: store as HF_TOKEN in Colab secrets)
hf_token = None
try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
    print("✓ Loaded HF_TOKEN from Colab secrets")
except Exception:
    pass

# Fallback: prompt
if not hf_token:
    hf_token = getpass("Paste your HuggingFace token (hf_...): ")

os.environ["HF_TOKEN"] = hf_token
os.environ["HUGGING_FACE_HUB_TOKEN"] = hf_token

from huggingface_hub import login
login(token=hf_token, add_to_git_credential=True)
print("✓ Logged in to HuggingFace")

## 1. Load Model

In [ ]:
import torch
import os
import gc
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

MODEL_NAME = "google/gemma-4-e4b"
MAX_SEQ_LENGTH = 1024  # A100 has plenty of room

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

print(f"Loading tokenizer for {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Loading model {MODEL_NAME} (this takes ~3-5 min)...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map={"": 0},          # Force everything on GPU 0 — avoids CPU offload OOMs
    dtype=torch.bfloat16,        # New API (replaces torch_dtype)
    attn_implementation="eager", # Required for Gemma 4 with bnb
)

model.config.use_cache = False
print(f"✓ Model loaded — {model.num_parameters():,} parameters")

gc.collect()
torch.cuda.empty_cache()
print(f"GPU memory used: {torch.cuda.memory_allocated()/1024**3:.2f} GB")
print(f"GPU memory free: {(torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated())/1024**3:.2f} GB")

## 2. Add LoRA Adapters

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

# Diagnostic: find non-standard Linear wrappers (Gemma 4 uses Gemma4ClippableLinear)
print("Scanning model for Linear wrappers...")
wrapper_classes = set()
target_module_names = set()
for name, module in model.named_modules():
    cls_name = module.__class__.__name__
    if "Linear" in cls_name and cls_name not in ("Linear", "Linear4bit", "Linear8bitLt"):
        wrapper_classes.add(cls_name)
    short_name = name.split(".")[-1]
    if short_name in {"q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"}:
        target_module_names.add(short_name)

print(f"  Wrapper classes found: {wrapper_classes or 'none'}")
print(f"  Target projection modules found: {target_module_names}")

# Unwrap any wrapper class that exposes the real layer via `.linear`
def unwrap_linear_wrappers(model, wrapper_class_names):
    if not wrapper_class_names:
        return model
    for name, module in list(model.named_modules()):
        if module.__class__.__name__ in wrapper_class_names:
            inner = getattr(module, "linear", None)
            if inner is None:
                continue
            parent_name = ".".join(name.split(".")[:-1])
            child_name = name.split(".")[-1]
            parent = model.get_submodule(parent_name) if parent_name else model
            setattr(parent, child_name, inner)
    return model

model = unwrap_linear_wrappers(model, wrapper_classes)

# Pick attention modules that exist in this model
target_modules = sorted([m for m in target_module_names if m in {"q_proj", "k_proj", "v_proj", "o_proj"}])
if not target_modules:
    raise RuntimeError(f"No standard target modules found! Available: {target_module_names}")
print(f"  Using target modules: {target_modules}")

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=target_modules,
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

gc.collect()
torch.cuda.empty_cache()
print(f"GPU memory after LoRA: {torch.cuda.memory_allocated()/1024**3:.2f} GB")

## 3. Prepare Dataset

In [ ]:
from datasets import load_dataset, Dataset

HINDI_SYSTEM = "तुम VidyaAI हो, एक बुद्धिमान शिक्षा सहायक। NCERT पाठ्यक्रम के अनुसार सरल हिंदी में जवाब दो। उदाहरणों का उपयोग करो।"
ENGLISH_SYSTEM = "You are VidyaAI, an intelligent education assistant. Answer according to NCERT curriculum in clear, simple English with examples."

def format_to_gemma(instruction, input_text, output_text):
    return (
        f"<start_of_turn>system\n{instruction}<end_of_turn>\n"
        f"<start_of_turn>user\n{input_text}<end_of_turn>\n"
        f"<start_of_turn>model\n{output_text}<end_of_turn>"
    )

# ── Source 1: ai4bharat Hindi instruction-following (Dolly Hindi) ──
print("Loading ai4bharat Hindi dolly...")
hindi_ds = load_dataset("ai4bharat/indic-instruct-data-v0.1", "dolly", split="hi[:2000]")
hindi_examples = []
for row in hindi_ds:
    q = (row.get("instruction", "") or "").strip()
    ctx = (row.get("context", "") or "").strip()
    a = (row.get("response", "") or "").strip()
    if q and a:
        full_q = f"{q}\n\nसंदर्भ: {ctx}" if ctx else q
        hindi_examples.append({"text": format_to_gemma(HINDI_SYSTEM, full_q, a)})
print(f"  ai4bharat Hindi: {len(hindi_examples)}")

# ── Source 2: CohereForAI/aya Hindi rows ──
print("Loading Aya Hindi...")
aya = load_dataset("CohereForAI/aya_dataset", split="train")
aya_hindi = aya.filter(lambda x: x["language_code"] == "hin")
n_aya = min(2000, len(aya_hindi))
aya_examples = []
for row in aya_hindi.select(range(n_aya)):
    q = (row.get("inputs", "") or "").strip()
    a = (row.get("targets", "") or "").strip()
    if q and a:
        aya_examples.append({"text": format_to_gemma(HINDI_SYSTEM, q, a)})
print(f"  Aya Hindi: {len(aya_examples)}")

# ── Source 3: SciQ English (parquet) ──
print("Loading SciQ...")
sciq = load_dataset("allenai/sciq", split="train[:2000]")
sciq_examples = []
for row in sciq:
    q = row.get("question", "").strip()
    a = row.get("correct_answer", "").strip()
    support = (row.get("support", "") or "")[:300].strip()
    if q and a:
        full_answer = f"{a}\n\nExplanation: {support}" if support else a
        sciq_examples.append({"text": format_to_gemma(ENGLISH_SYSTEM, q, full_answer)})
print(f"  SciQ: {len(sciq_examples)}")

# ── Source 4: Hand-crafted NCERT seed examples (high quality) ──
SEED = [
    (HINDI_SYSTEM, "प्रकाश संश्लेषण क्या है?",
     "प्रकाश संश्लेषण वह प्रक्रिया है जिसमें हरे पौधे सूर्य के प्रकाश की ऊर्जा का उपयोग करके कार्बन डाइऑक्साइड और पानी से ग्लूकोज बनाते हैं।\n\nसमीकरण: 6CO₂ + 6H₂O + सूर्य का प्रकाश → C₆H₁₂O₆ + 6O₂\n\nयह पत्तियों में क्लोरोफिल की मदद से होती है।"),
    (HINDI_SYSTEM, "पाइथागोरस प्रमेय क्या है?",
     "समकोण त्रिभुज में कर्ण² = आधार² + लंब²\n\nसूत्र: c² = a² + b²\n\nउदाहरण: भुजाएं 3, 4 cm → c² = 9+16 = 25 → c = 5 cm"),
    (HINDI_SYSTEM, "न्यूटन के गति के नियम बताइए।",
     "1. जड़त्व: वस्तु अवस्था बनाए रखती है जब तक बल न लगे।\n2. F = ma: बल = द्रव्यमान × त्वरण\n3. क्रिया-प्रतिक्रिया: हर क्रिया की बराबर विपरीत प्रतिक्रिया होती है।"),
    (HINDI_SYSTEM, "भारत की आज़ादी कब मिली?",
     "15 अगस्त 1947 को। गांधी जी के नेतृत्व में अहिंसा आंदोलन, 1942 भारत छोड़ो, और सुभाष बोस की आज़ाद हिंद फ़ौज ने स्वतंत्रता दिलाई।"),
    (HINDI_SYSTEM, "कोशिका क्या है?",
     "जीवन की सबसे छोटी इकाई।\n\nप्रोकैरियोटिक: बिना केंद्रक (बैक्टीरिया)\nयूकैरियोटिक: केंद्रक वाली (पौधे, जानवर)"),
    (HINDI_SYSTEM, "अम्ल और क्षार में अंतर?",
     "अम्ल: खट्टा, pH<7, नीला→लाल लिटमस। उदा: नींबू\nक्षार: कड़वा, pH>7, लाल→नीला लिटमस। उदा: साबुन\nमिलने पर: लवण + जल"),
    (ENGLISH_SYSTEM, "What is photosynthesis?",
     "Plants make glucose from CO₂ + H₂O using sunlight.\n6CO₂ + 6H₂O + Sunlight → C₆H₁₂O₆ + 6O₂\nHappens in chlorophyll-containing leaves."),
    (ENGLISH_SYSTEM, "Explain Newton's three laws.",
     "1. Inertia: objects stay at rest/motion unless forced.\n2. F=ma: force = mass × acceleration.\n3. Action-reaction: equal opposite forces."),
    (ENGLISH_SYSTEM, "What is democracy?",
     "Government by the people through free elections. Key: universal suffrage, rule of law, fundamental rights, independent judiciary. India is the largest democracy."),
    (ENGLISH_SYSTEM, "Difference between mitosis and meiosis?",
     "Mitosis: 2 identical diploid cells, for growth.\nMeiosis: 4 different haploid cells, for gametes.\nHumans: mitosis=46 chromosomes, meiosis=23."),
]
seed_examples = [{"text": format_to_gemma(s, i, o)} for s, i, o in SEED]
print(f"  Seed: {len(seed_examples)}")

all_examples = seed_examples + hindi_examples + aya_examples + sciq_examples
dataset = Dataset.from_list(all_examples).shuffle(seed=42)
print(f"\nTotal dataset: {len(dataset)} examples")
print(f"\nSample:\n{dataset[0]['text'][:300]}...")

## 4. Training

In [ ]:
from trl import SFTTrainer, SFTConfig

config = SFTConfig(
    output_dir="vidyaai-output",
    dataset_text_field="text",
    max_length=MAX_SEQ_LENGTH,        # NEW name in TRL 1.0+ (was max_seq_length)
    dataset_num_proc=2,
    packing=False,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,    # Effective batch size = 8
    warmup_steps=20,
    num_train_epochs=2,                # 2 epochs is enough for QLoRA
    learning_rate=2e-4,
    bf16=True,
    logging_steps=10,
    optim="paged_adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    seed=42,
    report_to="none",
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    save_strategy="no",                # Skip intermediate checkpoints, save manually at end
    dataloader_pin_memory=False,
)

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,        # NEW name in TRL 1.0+ (was tokenizer)
    train_dataset=dataset,
    args=config,
)
print("✓ Trainer initialized")
print(f"GPU memory before training: {torch.cuda.memory_allocated()/1024**3:.2f} GB")

In [ ]:
# Train!
trainer_stats = trainer.train()
print(f"Training completed in {trainer_stats.metrics['train_runtime']:.1f}s")
print(f"Final loss: {trainer_stats.metrics['train_loss']:.4f}")

## 5. Test the Fine-tuned Model

In [ ]:
model.eval()

test_questions = [
    "प्रकाश संश्लेषण क्या है?",
    "भारत के पहले प्रधानमंत्री कौन थे?",
    "What is the formula for area of a circle?",
    "गुरुत्वाकर्षण बल क्या है?",
]

for q in test_questions:
    prompt = (
        "<start_of_turn>system\n"
        "तुम VidyaAI हो, एक बुद्धिमान शिक्षा सहायक। NCERT पाठ्यक्रम के अनुसार सरल हिंदी में जवाब दो।"
        "<end_of_turn>\n"
        f"<start_of_turn>user\n{q}<end_of_turn>\n"
        "<start_of_turn>model\n"
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.1,
        )
    response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    print(f"\n{'='*60}")
    print(f"Q: {q}")
    print(f"A: {response}")

## 6. Save & Upload to HuggingFace

In [ ]:
# Save LoRA adapter locally
model.save_pretrained("vidyaai-gemma4-lora")
tokenizer.save_pretrained("vidyaai-gemma4-lora")
print("Saved LoRA adapter to vidyaai-gemma4-lora/")

In [ ]:
# Upload LoRA adapter to HuggingFace Hub (already logged in from cell 2)
HF_REPO = "yashv10k/vidyaai-gemma4-lora"

model.push_to_hub(HF_REPO)
tokenizer.push_to_hub(HF_REPO)
print(f"✓ Uploaded to https://huggingface.co/{HF_REPO}")